# 결손 데이터(결측치) 처리

### 결손데이터(결측치) 란?
- 결측치(Missing Data)는 값이 존재하지 않는 데이터를 의ㅣ
- 판다스에서는 NaN으로 표현

**결측치가 있다면 ? 어떻게 처리할것인가?**
1) 실무자와 협의한 후 진행 - 제거도 한 방법!
2) 평균으로 대체하는 값, 중앙값으로 대체하는 값 ..
    예) 날짜마다 온도가 저장되어 있는 데이터셋이라면..  
   20260601   20  
   ...  
   20260614   32  
   20260615   NaN        => **앞, 뒤의 값으로 대체**  
   20260616   31  
   ...  
   20260630   36  

In [3]:
import warnings
warnings.filterwarnings(action='ignore')

import pandas as pd
import numpy as np

### 1. 결손 데이터 생성

In [4]:
df = pd.DataFrame(
    {
        'Name' : ['Smith', 'Lee', 'Kim', 'Park'],
        'Age' : [20, np.nan, 30, 40],
        'Score' : [90, 85, np.nan, 75]
    }
)
df

,Name,Age,Score
0,Smith,20.0,90.0
1,Lee,NaN,85.0
2,Kim,30.0,NaN
3,Park,40.0,75.0


In [4]:
df.shape

(4, 3)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Name    4 non-null      object 
 1   Age     3 non-null      float64
 2   Score   3 non-null      float64
dtypes: float64(2), object(1)
memory usage: 228.0+ bytes


### 2. 결손 데이터 확인
- isna() = isnull()

In [8]:
df
df.isnull()

,Name,Age,Score
0,False,False,False
1,False,True,False
2,False,False,True
3,False,False,False


In [13]:
df.isna().sum()                # column 별로 NaN 값이 몇개 있는지
df.isna().sum().sum()          # df 전체에 NaN 값이 몇개 있는지

np.int64(2)

### 3. 결손 데이터 삭제
- dropna() : 결손 데이터가 포함된 행 또는 열 삭제

In [15]:
filter_na = df.dropna()
filter_na

,Name,Age,Score
0,Smith,20.0,90.0
3,Park,40.0,75.0


In [18]:
# 결측치 데이터가 포함된 열 삭제
filter_na_cols = df.dropna(axis = 1)          # NaN 값이 들어있는 column drop
filter_na_cols

,Name
0,Smith
1,Lee
2,Kim
3,Park


In [22]:
# 결측치가 저장된 특정 행 삭제
df.dropna(axis = 0, subset=['Age'])

,Name,Age,Score
0,Smith,20.0,90.0
2,Kim,30.0,NaN
3,Park,40.0,75.0


In [23]:
# Addr 컬럼 추가, 값은 모두 결측치 (NaN)
df['Addr'] = np.nan
df

,Name,Age,Score,Addr
0,Smith,20.0,90.0,NaN
1,Lee,NaN,85.0,NaN
2,Kim,30.0,NaN,NaN
3,Park,40.0,75.0,NaN


In [24]:
df.dropna(how = 'all', axis=1, inplace = True)
df

,Name,Age,Score
0,Smith,20.0,90.0
1,Lee,NaN,85.0
2,Kim,30.0,NaN
3,Park,40.0,75.0


In [25]:
# 하나라도 결측치를 포함한 경우 삭제 how = 'any'
df.dropna(how = 'any', axis = 0)

,Name,Age,Score
0,Smith,20.0,90.0
3,Park,40.0,75.0


### 4. 결손 데이터 대체
- fillna() : 결손 데이터를 특정값으로 대체

In [9]:
# 결손데이터를 0으로 대체
df_str = df.fillna('Unknown')
df_str.info()                  # data type이 전부 object으로 바뀜

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Name    4 non-null      object
 1   Age     4 non-null      object
 2   Score   4 non-null      object
dtypes: object(3)
memory usage: 228.0+ bytes


In [12]:
# Age 컬럼의 결손 데이터만 0으로 대체
df[['Age', 'Score']].fillna(0)             # 실행한 col만 출력됨

,Age,Score
0,20.0,90.0
1,0.0,85.0
2,30.0,0.0
3,40.0,75.0


### 5. 평균값으로 대체

In [20]:
df = pd.DataFrame(
    {
        'Name' : ['Smith', 'Lee', 'Kim', 'Park'],
        'Age' : [20, np.nan, 20, 40],
        'Score' : [90, 85, np.nan, 75]
    }
)
df

,Name,Age,Score
0,Smith,20.0,90.0
1,Lee,NaN,85.0
2,Kim,20.0,NaN
3,Park,40.0,75.0


In [16]:
# Age 컬럼의 결손 데이터를 평균 값으로 대체
age_mean = df['Age'].mean()
df['Age'] = df['Age'].fillna(age_mean)
df

,Name,Age,Score
0,Smith,20.0,90.0
1,Lee,30.0,85.0
2,Kim,30.0,NaN
3,Park,40.0,75.0


### 6. 최빈값으로 대체
- 가장 많이 등장하는 값으로 대체
- model() : 가장 많이 등장한 값 반환

In [21]:
# Age 컬럼의 최빈값 조회
df['Age'].mode()

0    20.0
Name: Age, dtype: float64

,Name,Age,Score
0,Smith,20.0,90.0
1,Lee,NaN,85.0
2,Kim,30.0,NaN
3,Park,40.0,75.0


### Titanic 데이터 결측치 처리

In [23]:
# 데이터 로드
titanic_df = pd.read_csv('data/titanic/train.csv')
titanic_df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [25]:
titanic_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [37]:
# 결손 데이터 수 확인
titanic_df.isna().sum()

PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Cabin          0
Embarked       2
dtype: int64

In [31]:
# Cabin 컬럼의 687개의 결측 데이터는 '0000'로 대체
titanic_df['Cabin'] = titanic_df['Cabin'].fillna('0000')
titanic_df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,0000,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,0000,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,0000,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,0000,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,0000,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [38]:
# 'Age' 컬럼은 평균값을 구해 결측 데이터를 대체
age_mean = titanic_df['Age'].mean().round()
titanic_df['Age'] = titanic_df['Age'].fillna(age_mean)
titanic_df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,0000,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,0000,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,0000,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,0000,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,30.0,1,2,W./C. 6607,23.4500,0000,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [41]:
#  Embarked의 결측치를 최빈값으로 대체
most_embarked = titanic_df['Embarked'].mode()
most_embarked

0    S
Name: Embarked, dtype: object

In [42]:
titanic_df['Embarked'] = titanic_df['Embarked'].fillna(most_embarked[0])
titanic_df.isna().sum()

PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Cabin          0
Embarked       0
dtype: int64

### 10. nunique & replace

#### (1) nunique

In [46]:
# value_counts() => 범주형 데이터 조회
titanic_df['Embarked'].value_counts()

Embarked
S    646
C    168
Q     77
Name: count, dtype: int64

In [44]:
titanic_df.nunique()

PassengerId    891
Survived         2
Pclass           3
Name           891
Sex              2
Age             88
SibSp            7
Parch            7
Ticket         681
Fare           248
Cabin          148
Embarked         3
dtype: int64

#### (2) replace
- 원본값을 다른값으로 대체

In [48]:
# Cabin의 '0000' 값을 'C000' 값으로 대체
titanic_df['Cabin'] = titanic_df['Cabin'].replace('0000', 'C000')
titanic_df['Cabin']

0      C000
1       C85
2      C000
3      C123
4      C000
       ... 
886    C000
887     B42
888    C000
889    C148
890    C000
Name: Cabin, Length: 891, dtype: object